In [ ]:
"""
Sample Script: extract_data.py
Confidentiality Notice: This script is a sample representation and contains no actual client data.
"""

from pyspark.sql import SparkSession
from pyspark.sql.functions import input_file_name

In [ ]:
# Initialize Spark Session with performance config (optional)
spark = SparkSession.builder \
    .appName("ExtractSAPData") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.files.ignoreCorruptFiles", "true") \
    .getOrCreate()

In [ ]:
# Configurable paths (for demo)
source_path = "dbfs:/mnt/source_data/"
file_type = "parquet"  # or 'json'

# Optimization: Pushdown + column pruning example
required_columns = ["id", "name", "status", "region", "amount"]

In [ ]:
# Read data with optimizations
if file_type == "parquet":
    df = spark.read.parquet(source_path).select(*required_columns)
elif file_type == "json":
    df = spark.read.option("multiline", "true").json(source_path).select(*required_columns)
else:
    raise ValueError("Unsupported file type: choose 'parquet' or 'json'")

In [ ]:
# Add metadata
df = df.withColumn("source_file", input_file_name())

# Optimization: Cache if reused downstream (e.g., for profiling or sampling)
df.cache()

In [ ]:
# Write optimized Delta with partitioning (if column makes sense)
df.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("region") \
    .save("dbfs:/mnt/bronze_layer/sample_data")

spark.stop()